In [ ]:
import json
from pathlib import Path

In [ ]:
DELHI_ALL_DATA = Path(
    "/Users/hariomnarang/Desktop/personal/roads/mapillary_downloader/data/delhi/images"
)
CHUNKS_DEST = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks"
)

In [ ]:
from mtrain.utils import globL, mkdir
from itertools import batched
import shutil

images = globL(DELHI_ALL_DATA, "*.jpg")
len(images)

In [ ]:
from tqdm import tqdm

chunk_size = 100
batches = list(batched(images[:10], chunk_size))
for i, batch in enumerate(tqdm(batches)):
    for image_path in batch:
        chunk_dest = mkdir(CHUNKS_DEST / "lmao")
        dest = mkdir(chunk_dest / image_path.stem)
        shutil.copy(image_path, dest / "image.jpg")

In [ ]:
# test uploaded input
TEST_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi/chunks/lmao"
)

In [ ]:
from mtrain.example_dir.core import load_npz
from mtrain.example_dir.defaults.smallnet import default_smallnet_learners
from mtrain.example_dir.defaults.negmask import default_negmask_learners 
from mtrain.example_dir import ExampleDir
from mtrain.example_dir.iterdir import get_dirs
from mtrain.utils import *
from mtrain.utils import overlay_mask_on_img as OV

negmask = default_negmask_learners(
    Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"),
    ["md", "high-recall", "unblurred"],
    4,
)

smallnet = default_smallnet_learners(
    Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "sm"], 4
)

ds = list(get_dirs(TEST_DIR))
ds = [d for d in ds if d.name in ["1018919538513111", "1638730477092176", "1974648526381662", "295307108848223"]]

edir = ExampleDir(
    ds[0], smallnet, negmask
)

image = DiskImage.load(edir.image_path)
mdo, mdt = edir.negmask_paths("md", "md", False)
mdo, mdt = load_npz(mdo), load_npz(mdt)

hro, hrt = edir.negmask_paths("high-recall", "md")
hro, hrt = load_npz(hro), load_npz(hrt)

unbo, unbt = edir.negmask_paths("unblurred", "sm")
unbo, unbt = load_npz(unbo), load_npz(unbt)

show([
    image,
    OV(image, DiskBooleanMask.load(edir.trimmed_mask_path("md"))),
    OV(image, mdt > mdo),
    OV(image, hrt > hro),
    OV(image, unbt > unbo),
])
